# KnowMate 수집기 검증 - Step 2: Orphan 안전장치
## 폴더 루트 가드 / 대량삭제 차단(30%) / Soft Delete / Dry-run

In [ ]:
import os
import json
import uuid
import logging
from pathlib import Path
from datetime import datetime

# 로거 설정
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s - %(message)s')
logger = logging.getLogger('knowmate.collector')

# 설정값 (config.yaml 대신 여기서 직접 정의)
CONFIG = {
    'max_delete_ratio': 0.30,  # 30% 초과 시 차단
    'dry_run': True,           # 기본값 True
}

# 테스트용 경로
WATCH_FOLDER = Path('./test_docs')
STATE_FILE   = Path('./index_state.json')

# Step1에서 만든 유틸 함수 재사용
def load_state(state_file: Path) -> dict:
    """저장된 스캔 상태 로드"""
    if not state_file.exists():
        return {}
    with open(state_file, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_state(state_file: Path, state: dict) -> None:
    """현재 스캔 상태 저장"""
    with open(state_file, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

print('설정 로드 완료')
print(f"max_delete_ratio: {CONFIG['max_delete_ratio']*100:.0f}%")
print(f"dry_run: {CONFIG['dry_run']}")

## 테스트 환경 준비
Step1 state.json이 있다고 가정하고 mock 데이터 세팅

In [ ]:
# 테스트 폴더 + 파일 생성
WATCH_FOLDER.mkdir(exist_ok=True)
(WATCH_FOLDER / 'A.txt').write_text('문서 A', encoding='utf-8')
(WATCH_FOLDER / 'C.txt').write_text('문서 C', encoding='utf-8')
# B.txt는 이미 삭제된 상태 (Step1에서 삭제했음)

# mock state.json: A, B, C 모두 인덱스에 있는 상태로 세팅
mock_state = {
    str(WATCH_FOLDER / 'A.txt'): {
        'mtime': 1000.0, 'size': 100,
        'indexed_at': '2026-06-01T09:00:00',
        'chunk_ids': [str(uuid.uuid4()), str(uuid.uuid4())]
    },
    str(WATCH_FOLDER / 'B.txt'): {  # ← orphan 대상
        'mtime': 2000.0, 'size': 200,
        'indexed_at': '2026-06-01T09:00:00',
        'chunk_ids': [str(uuid.uuid4())]
    },
    str(WATCH_FOLDER / 'C.txt'): {
        'mtime': 3000.0, 'size': 300,
        'indexed_at': '2026-06-01T09:00:00',
        'chunk_ids': [str(uuid.uuid4())]
    },
}
save_state(STATE_FILE, mock_state)
print('mock state.json 세팅 완료')
print(f'인덱스 총 {len(mock_state)}건: A.txt, B.txt(orphan 대상), C.txt')

## 1. 폴더 루트 가드
감시 폴더가 접근 불가하면 해당 폴더 정리 전체 스킵

In [ ]:
def check_folder_accessible(folder: Path) -> bool:
    """폴더 루트 접근 가능 여부 확인"""
    if not folder.exists():
        logger.warning(f'[루트가드] 폴더 없음 - 정리 스킵: {folder}')
        return False
    if not os.access(folder, os.R_OK):
        logger.warning(f'[루트가드] 폴더 접근 불가 - 정리 스킵: {folder}')
        return False
    return True

# 정상 폴더 테스트
print('=== 정상 폴더 ===' )
result = check_folder_accessible(WATCH_FOLDER)
print(f'접근 가능: {result}')

# 존재하지 않는 폴더 테스트 (네트워크 드라이브 끊김 시뮬레이션)
print('\n=== 존재하지 않는 폴더 (네트워크 단절 시뮬레이션) ===')
result = check_folder_accessible(Path('Z:/공유폴더/없는경로'))
print(f'접근 가능: {result}')
print('\n✅ 기대값: 정상=True, 없는폴더=False + WARNING 로그')

## 2. Orphan 목록 추출
인덱스에 있으나 디스크에 없는 파일 찾기

In [ ]:
def find_orphans(state: dict, folder: Path) -> list[str]:
    """폴더 소속 항목 중 디스크에 없는 파일(orphan) 목록 반환"""
    folder_str = str(folder)
    orphans = [
        path for path in state.keys()
        if path.startswith(folder_str) and not Path(path).exists()
        and 'deleted_at' not in state[path]  # 이미 마킹된 건 제외
    ]
    return orphans

state = load_state(STATE_FILE)
orphans = find_orphans(state, WATCH_FOLDER)

print(f'인덱스 총 건수: {len(state)}건')
print(f'Orphan 발견: {len(orphans)}건')
for o in orphans:
    print(f'  → {Path(o).name}')
print('\n✅ 기대값: B.txt 1건')

## 3. 대량 삭제 차단기 (30% 룰)

In [ ]:
def check_delete_ratio(state: dict, folder: Path, orphans: list[str], max_ratio: float) -> bool:
    """
    orphan 비율이 max_ratio 초과 시 False 반환 (정리 중단)
    폴더 단위 스코프로 비율 계산
    """
    folder_str = str(folder)
    folder_total = sum(1 for p in state if p.startswith(folder_str))
    if folder_total == 0:
        return True
    ratio = len(orphans) / folder_total
    if ratio > max_ratio:
        logger.error(
            f'[대량삭제차단] orphan 비율 {ratio*100:.1f}% > 임계값 {max_ratio*100:.0f}% '
            f'→ 폴더 정리 중단: {folder}'
        )
        return False
    logger.info(f'[비율체크] orphan 비율 {ratio*100:.1f}% → 정리 진행')
    return True

# 정상 케이스 (1/3 = 33% → 30% 초과이므로 차단됨)
print('=== 현재 케이스: 3건 중 1건 orphan (33%) ===')
ok = check_delete_ratio(state, WATCH_FOLDER, orphans, CONFIG['max_delete_ratio'])
print(f'정리 진행 가능: {ok}')

# 임계값 50%로 올려서 통과 케이스
print('\n=== 임계값 50%로 완화 시 ===')
ok = check_delete_ratio(state, WATCH_FOLDER, orphans, 0.50)
print(f'정리 진행 가능: {ok}')
print('\n✅ 기대값: 30%→차단(False), 50%→통과(True)')

## 4. Soft Delete — 1차 마킹
orphan 즉시 삭제 X → deleted_at 마킹 후 검색에서만 제외

In [ ]:
def soft_delete_mark(state: dict, orphans: list[str], dry_run: bool) -> dict:
    """orphan에 deleted_at 마킹. dry_run=True면 로그만 출력"""
    now = datetime.now().isoformat()
    for path in orphans:
        if dry_run:
            logger.info(f'[DRY-RUN] soft delete 마킹 예정: {Path(path).name}')
        else:
            state[path]['deleted_at'] = now
            logger.info(f'[SOFT-DELETE] 마킹 완료: {Path(path).name} → {now}')
    return state

# dry_run=True 테스트
print('=== dry_run=True (로그만 출력) ===')
state = load_state(STATE_FILE)
state = soft_delete_mark(state, orphans, dry_run=True)
print(f"B.txt deleted_at 존재: {'deleted_at' in state.get(str(WATCH_FOLDER/'B.txt'), {})}")

# dry_run=False 테스트
print('\n=== dry_run=False (실제 마킹) ===')
state = soft_delete_mark(state, orphans, dry_run=False)
save_state(STATE_FILE, state)
print(f"B.txt deleted_at 존재: {'deleted_at' in state.get(str(WATCH_FOLDER/'B.txt'), {})}")
print(f"B.txt deleted_at 값: {state.get(str(WATCH_FOLDER/'B.txt'), {}).get('deleted_at')}")
print('\n✅ 기대값: dry_run=True→마킹없음, False→deleted_at 세팅됨')

## 5. Soft Delete — 2차 확인 후 물리삭제
2회 연속 디스크에 없는 경우에만 실제 삭제 (chunk_ids 반환)

In [ ]:
def physical_delete(state: dict, folder: Path, dry_run: bool) -> tuple[dict, list[str]]:
    """
    deleted_at 마킹된 항목 중 이번 스캔에서도 디스크에 없으면 물리삭제
    반환: (갱신된 state, 물리삭제된 chunk_ids 목록)
    """
    folder_str = str(folder)
    to_delete  = [
        path for path, info in state.items()
        if path.startswith(folder_str)
        and 'deleted_at' in info
        and not Path(path).exists()  # 이번에도 없음 = 2회 연속
    ]
    deleted_chunk_ids = []
    for path in to_delete:
        deleted_chunk_ids.extend(state[path].get('chunk_ids', []))
        if dry_run:
            logger.info(f'[DRY-RUN] 물리삭제 예정: {Path(path).name} '
                       f'chunk_ids={len(state[path].get("chunk_ids",[]))}건')
        else:
            logger.info(f'[물리삭제] state에서 제거: {Path(path).name}')
            del state[path]
    return state, deleted_chunk_ids

# 2차 스캔 시뮬레이션 (B.txt는 여전히 디스크에 없음)
print('=== 2차 스캔: B.txt 여전히 없음 → 물리삭제 ===')
state = load_state(STATE_FILE)
state, deleted_chunks = physical_delete(state, WATCH_FOLDER, dry_run=False)
save_state(STATE_FILE, state)

print(f'\n남은 인덱스: {len(state)}건')
print(f'삭제된 chunk_ids: {len(deleted_chunks)}건')
for cid in deleted_chunks:
    print(f'  → LanceDB 삭제 대상: {cid}')
print('\n✅ 기대값: 인덱스 2건(A,C), chunk_ids 1건 반환')

## 6. 파일 부활 시나리오
deleted_at 마킹 후 파일이 다시 생기면 마킹 해제

In [ ]:
def revive_if_exists(state: dict) -> dict:
    """deleted_at 마킹됐지만 디스크에 다시 나타난 파일 마킹 해제"""
    revived = []
    for path, info in state.items():
        if 'deleted_at' in info and Path(path).exists():
            del info['deleted_at']
            revived.append(Path(path).name)
            logger.info(f'[부활] deleted_at 마킹 해제: {Path(path).name}')
    if revived:
        print(f'부활 파일: {revived}')
    else:
        print('부활 파일 없음')
    return state

# 시뮬레이션: B.txt mock 재등록 후 부활 테스트
state = load_state(STATE_FILE)
b_path = str(WATCH_FOLDER / 'B.txt')
state[b_path] = {
    'mtime': 9999.0, 'size': 999,
    'indexed_at': '2026-06-20T00:00:00',
    'chunk_ids': [str(uuid.uuid4())],
    'deleted_at': '2026-06-20T10:00:00'  # 마킹된 상태
}
(WATCH_FOLDER / 'B.txt').write_text('B 부활', encoding='utf-8')  # 파일 다시 생성

print('=== B.txt 부활 시나리오 ===')
state = revive_if_exists(state)
print(f"B.txt deleted_at 제거됨: {'deleted_at' not in state.get(b_path, {})}")
print('\n✅ 기대값: deleted_at 제거됨=True')

## 7. 전체 orphan 정리 사이클 통합
위 함수들을 하나의 흐름으로 연결

In [ ]:
def run_cleanup_cycle(folder: Path, state_file: Path, config: dict) -> dict:
    """
    orphan 정리 전체 사이클
    반환: 사이클 리포트 dict
    """
    report = {
        'folder': str(folder),
        'skipped': False,
        'orphans_marked': 0,
        'physically_deleted': 0,
        'chunk_ids_to_remove': [],
        'blocked_by_ratio': False,
    }

    # 1. 폴더 루트 가드
    if not check_folder_accessible(folder):
        report['skipped'] = True
        return report

    state = load_state(state_file)

    # 2. 부활 파일 마킹 해제
    state = revive_if_exists(state)

    # 3. orphan 목록 추출
    orphans = find_orphans(state, folder)

    # 4. 대량 삭제 차단
    if not check_delete_ratio(state, folder, orphans, config['max_delete_ratio']):
        report['blocked_by_ratio'] = True
        return report

    # 5. soft delete 마킹
    state = soft_delete_mark(state, orphans, config['dry_run'])
    report['orphans_marked'] = len(orphans)

    # 6. 2회 확인 물리삭제
    state, deleted_chunks = physical_delete(state, folder, config['dry_run'])
    report['physically_deleted'] = len(deleted_chunks)
    report['chunk_ids_to_remove'] = deleted_chunks

    if not config['dry_run']:
        save_state(state_file, state)

    # 7. 사이클 리포트 로그
    logger.info(
        f"[사이클리포트] 폴더={folder} "
        f"마킹={report['orphans_marked']} "
        f"물리삭제={report['physically_deleted']} "
        f"차단={report['blocked_by_ratio']}"
    )
    return report

# 전체 사이클 실행 (임계값 50%로 통과되게)
config_test = {'max_delete_ratio': 0.50, 'dry_run': False}
print('=== 전체 orphan 정리 사이클 ===')
report = run_cleanup_cycle(WATCH_FOLDER, STATE_FILE, config_test)
print('\n--- 사이클 리포트 ---')
for k, v in report.items():
    print(f'  {k}: {v}')

## 정리
- `check_folder_accessible()` : 폴더 루트 가드
- `find_orphans()` : orphan 목록 추출
- `check_delete_ratio()` : 30% 대량삭제 차단
- `soft_delete_mark()` : 1차 마킹 (deleted_at)
- `physical_delete()` : 2회 확인 후 물리삭제 + chunk_ids 반환
- `revive_if_exists()` : 파일 부활 시 마킹 해제
- `run_cleanup_cycle()` : 전체 사이클 통합

**다음 Step 3**: 스케줄러 (유휴시간 감지 + 증분 큐 우선순위)